---
title: Return Statistics
execute:
  enabled: true
---

This page's code cells are **executed live** by Quarto (via Jupyter) every time the
docs are built. `q.stats` computes return-stream statistics with no Plotly
dependency; `q.plot` renders them. See the [Interactive Demo](demo.ipynb) for
feature engineering and price-chart examples.

## Synthetic return streams

We simulate a daily strategy and benchmark return series with a random walk so this
page has no network dependency:

In [1]:
import numpy as np
import pandas as pd

import qrt as q

rng = np.random.default_rng(7)
dates = pd.bdate_range("2022-01-03", "2025-12-31")

benchmark = pd.Series(rng.normal(loc=0.0004, scale=0.010, size=len(dates)), index=dates, name="SPY")
noise = rng.normal(loc=0.0002, scale=0.006, size=len(dates))
strategy = (1.3 * benchmark + noise).rename("Strategy")

pd.concat([strategy, benchmark], axis=1).tail()

,Strategy,SPY
2025-12-25,0.014558,0.006233
2025-12-26,0.007899,-0.002061
2025-12-29,0.005932,0.008720
2025-12-30,-0.001641,-0.000037
2025-12-31,0.029661,0.017809


## Headline performance

`q.stats.performance` returns a pandas `Series` of Total Return, CAGR, Volatility,
Sharpe, Sortino, Calmar, Max Drawdown, Win Rate, and Periods. The annualization
frequency is inferred from the index (252 for this daily series) unless
`periods_per_year` is given explicitly:

In [2]:
q.stats.performance(strategy)

Total Return      -0.314637
CAGR              -0.087240
Volatility         0.218642
Sharpe            -0.308251
Sortino           -0.435830
Calmar            -0.163264
Max Drawdown      -0.534350
Win Rate           0.493768
Periods         1043.000000
Name: Strategy, dtype: float64

## Benchmark-relative: alpha and beta

`q.stats.beta` measures sensitivity to the benchmark; `q.stats.alpha` is the
annualized Jensen alpha unexplained by that beta exposure. `q.stats.benchmark_stats`
combines both with active return, correlation, tracking error, and information
ratio:

In [3]:
print(f"Beta: {q.stats.beta(strategy, benchmark):.2f}")
print(f"Alpha: {q.stats.alpha(strategy, benchmark):.2%}")

q.stats.benchmark_stats(strategy, benchmark)

Beta: 1.30
Alpha: 4.02%


Strategy Total Return       -0.314637
Benchmark Total Return      -0.321604
Active Return                0.010269
Beta                         1.304278
Alpha                        0.040183
Correlation                  0.895104
Tracking Error               0.107646
Information Ratio            0.140143
Periods                   1043.000000
Name: Strategy vs SPY, dtype: float64

## Rolling diagnostics

Rolling metrics make regime changes visible. A 63-business-day window is roughly
one quarter. The four diagnostics are combined into one interactive figure with
`q.plot.col`:

In [4]:
window = 63
rolling = pd.concat(
    {
        "Volatility": q.stats.rolling_volatility(strategy, window),
        "Sharpe": q.stats.rolling_sharpe(strategy, window),
        "Beta": q.stats.rolling_beta(strategy, benchmark, window),
        "Alpha": q.stats.rolling_alpha(strategy, benchmark, window),
    },
    axis=1,
)

fig = q.plot.col(
    rolling,
    title=f"{window}-day rolling risk and benchmark diagnostics",
    ylabel="Value",
    height=550,
)
fig.show()

## Calendar returns

`q.stats.monthly_returns` compounds daily returns within each calendar month into a
year-by-month table; `q.plot.monthly_heatmap` renders it as an interactive heatmap:

In [5]:
display(q.stats.monthly_returns(strategy).style.format("{:.1%}"))

fig = q.plot.monthly_heatmap(strategy, title="Strategy monthly returns")
fig.show()

Month,1,2,3,4,5,6,7,8,9,10,11,12
Year,,,,,,,,,,,,
2022,-11.4%,-10.4%,6.0%,-0.6%,-0.9%,-9.2%,-2.9%,-5.1%,2.3%,4.8%,-4.2%,-6.1%
2023,0.1%,12.8%,-7.1%,6.5%,-0.5%,2.8%,-5.7%,-0.4%,-4.7%,-6.3%,-1.2%,-5.2%
2024,-10.2%,-5.0%,-3.1%,-1.6%,3.1%,-1.6%,9.3%,-11.6%,19.1%,-7.7%,-0.8%,8.3%
2025,-11.9%,3.9%,-10.0%,8.6%,16.1%,4.4%,2.1%,6.4%,4.6%,0.1%,2.8%,-4.3%


## Chained API: `q.stats.returns(...)`

For notebook exploration, `q.stats.returns()` binds a return stream (and optional
benchmark) once and exposes the same stats as methods, plus `.plot(kind=...)` which
delegates to `q.plot`. Each call creates a fresh, independent object — there is no
hidden global state to reason about:

In [6]:
bound = q.stats.returns(strategy, benchmark=benchmark)

bound.performance()

Total Return      -0.314637
CAGR              -0.087240
Volatility         0.218642
Sharpe            -0.308251
Sortino           -0.435830
Calmar            -0.163264
Max Drawdown      -0.534350
Win Rate           0.493768
Periods         1043.000000
Name: Strategy, dtype: float64

`kind` accepts `"performance"`/`"tearsheet"` (equity + drawdown report), `"equity"`,
`"drawdown"`, or `"monthly_heatmap"`. The bound benchmark is passed through
automatically for the report variants:

In [7]:
print(f"Beta: {bound.beta():.2f}")
print(f"Alpha: {bound.alpha():.2%}")

fig = bound.plot("performance")
fig.show()

Beta: 1.30
Alpha: 4.02%
